# 🔄 Grundlegende Agenten-Workflows mit Microsoft Foundry (Python)

## 📋 Workflow-Orchestrierungsanleitung

Dieses Notebook stellt die leistungsstarken **Workflow Builder**-Funktionen des Microsoft Agent Frameworks vor. Lernen Sie, wie Sie anspruchsvolle, mehrstufige Agenten-Workflows erstellen, die komplexe Geschäftsprozesse bewältigen und mehrere KI-Operationen nahtlos koordinieren können.

> **Migrationshinweis:** Dieses Beispiel hat zuvor auf GitHub Models verwiesen. GitHub Models ist veraltet (Einstellung im Juli 2026), daher wird jetzt **Microsoft Foundry** über den `FoundryChatClient` genutzt, der die Azure OpenAI **Responses API** anspricht.

## 🎯 Lernziele

### 🏗️ **Workflow-Architektur**
- **Workflow Builder**: Entwurf und Orchestrierung komplexer mehrstufiger Prozesse
- **Ereignisgesteuerte Ausführung**: Handhabung von Workflow-Ereignissen und Zustandsübergängen
- **Visuelles Workflow-Design**: Erstellung und Visualisierung von Workflow-Strukturen
- **Microsoft Foundry Integration**: Nutzung von KI-Modellen innerhalb von Workflow-Kontexten

### 🔄 **Prozessorchestrierung**
- **Sequentielle Operationen**: Kettung mehrerer Agentenaufgaben in logischer Reihenfolge
- **Bedingte Logik**: Implementierung von Entscheidungspunkten und verzweigten Workflows
- **Fehlerbehandlung**: Robuste Fehlererholung und Workflow-Resilienz
- **Zustandsverwaltung**: Verfolgung und Verwaltung des Workflow-Ausführungszustands

### 📊 **Enterprise Workflow-Muster**
- **Geschäftsprozessautomatisierung**: Automatisierung komplexer organisatorischer Workflows
- **Multi-Agenten-Koordination**: Koordination mehrerer spezialisierter Agenten
- **Skalierbare Ausführung**: Konzeption von Workflows für unternehmensweite Prozesse
- **Überwachung & Beobachtbarkeit**: Verfolgung der Workflow-Leistung und Ergebnisse

## ⚙️ Voraussetzungen & Einrichtung

### 📦 **Benötigte Abhängigkeiten**

Installieren Sie das Agent Framework mit Workflow-Funktionalitäten:

```bash
pip install agent-framework -U
```

### 🔑 **Microsoft Foundry Konfiguration**

Melden Sie sich mit der Azure CLI (`az login`) an, damit `AzureCliCredential` sich authentifizieren kann, und legen Sie anschließend Ihre Microsoft Foundry-Projektdetails fest.

**Umgebungseinrichtung (.env-Datei):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

### 🏢 **Enterprise-Anwendungsfälle**

**Beispiele für Geschäftsprozesse:**
- **Kunden-Onboarding**: Mehrstufige Verifizierungs- und Einrichtungsworkflows
- **Content-Pipeline**: Automatisierte Inhaltserstellung, Prüfung und Veröffentlichung
- **Datenverarbeitung**: ETL-Workflows mit KI-gestützter Transformation
- **Qualitätssicherung**: Automatisierte Test- und Validierungsprozesse

**Vorteile von Workflows:**
- 🎯 **Zuverlässigkeit**: Deterministische Ausführung mit Fehlererholung
- 📈 **Skalierbarkeit**: Bewältigung von Volumenstarker Prozessautomatisierung
- 🔍 **Beobachtbarkeit**: Lückenlose Audit-Trails und Überwachung
- 🔧 **Wartbarkeit**: Visuelles Design und modulare Komponenten

## 🎨 Workflow-Entwurfsmuster

### Grundlegende Workflow-Struktur
```mermaid
graph TD
    A[Start] --> B[Aufgaben des Agenten 1]
    B --> C{Entscheidungspunkt}
    C -->|Erfolg| D[Aufgaben des Agenten 2]
    C -->|Fehler| E[Fehlerbehandlung]
    D --> F[Ende]
    E --> F
```

**Wichtige Komponenten:**
- **WorkflowBuilder**: Haupt-Orchestrierungs-Engine
- **WorkflowEvent**: Ereignisverwaltung und Kommunikation
- **WorkflowViz**: Visuelle Workflow-Darstellung und Debugging

Lassen Sie uns Ihren ersten intelligenten Workflow erstellen! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
